# 01 — Gerar pares real/manipulado por grupo

Adaptação exploratória de composição facial (blending), **não reprodução integral** de Face X-Ray ou de Trinh e Liu. Não se trata de geração por GAN. O experimento mede o detector para esta manipulação; não demonstra generalização para todas as deepfakes. Fonte e alvo pertencem ao mesmo grupo anotado e ao mesmo split. A imagem manipulada herda os metadados do alvo, não uma nova inferência de atributos.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/tcc_justica')
ROOT.mkdir(parents=True, exist_ok=True)
print('Pasta persistente:', ROOT)


In [ ]:
%pip -q install dlib opencv-python-headless scipy tqdm
import numpy as np, pandas as pd, cv2, dlib, hashlib, json
from scipy.spatial import cKDTree
from tqdm.auto import tqdm
import urllib.request,bz2
predictor=ROOT/'shape_predictor_68_face_landmarks.dat'
if not predictor.exists():
    packed=ROOT/'landmarks.dat.bz2'
    urllib.request.urlretrieve('https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2',packed)
    with bz2.open(packed,'rb') as src, open(predictor,'wb') as dst:
        import shutil;shutil.copyfileobj(src,dst)


## 1. Funções de alinhamento e composição
Reaproveitamos a estrutura do gerador enviado, com rejeição de fotos sem um único rosto e registro de falhas. Não há fallback que rotule uma imagem original como falsa.


In [ ]:
class LandmarkExtractor:
    """Detecta rosto + 68 landmarks faciais usando dlib."""

    def __init__(self, predictor_path: str):
        if dlib is None:
            raise ImportError(
                "dlib não está instalado. Rode: "
                "pip install --break-system-packages dlib"
            )
        if not os.path.exists(predictor_path):
            raise FileNotFoundError(
                f"Não encontrei o predictor em {predictor_path}. "
                "Baixe shape_predictor_68_face_landmarks.dat (veja o "
                "cabeçalho deste arquivo)."
            )
        self.detector = dlib.get_frontal_face_detector()
        self.predictor = dlib.shape_predictor(predictor_path)

    def extract(self, image_bgr: np.ndarray):
        """Retorna (landmarks[68,2], bbox) do maior rosto encontrado,
        ou (None, None) se nenhum rosto for detectado."""
        gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
        faces = self.detector(gray, 1)
        if len(faces) == 0:
            return None, None

        # pega o maior rosto detectado (mais provável de ser o principal)

        if len(faces) != 1:
            return None, None
        face = faces[0]
        shape = self.predictor(gray, face)
        landmarks = np.array([[p.x, p.y] for p in shape.parts()], dtype=np.float64)
        bbox = (face.left(), face.top(), face.right(), face.bottom())
        return landmarks, bbox


def normalize_landmarks(landmarks: np.ndarray) -> np.ndarray:
    """Centraliza e normaliza escala dos landmarks para que a comparação
    de distância entre rostos de tamanhos/posições diferentes seja justa."""
    centroid = landmarks.mean(axis=0)
    centered = landmarks - centroid
    scale = np.sqrt((centered ** 2).sum(axis=1)).mean()
    if scale < 1e-6:
        scale = 1.0
    return (centered / scale).flatten()


# --------------------------------------------------------------------------
# 2. Indexação por subgrupo (raça, gênero) + busca do rosto mais próximo
# --------------------------------------------------------------------------

class SubgroupIndex:
    """Para cada subgrupo (raça, gênero), guarda os landmarks normalizados
    de todas as imagens e permite buscar o vizinho mais próximo via KD-Tree."""

    def __init__(self):
        self.by_subgroup = {}  # (race, gender) -> {"paths":[...], "vecs": np.array, "tree": cKDTree}

    def add(self, path: str, race: str, gender: str, norm_vec: np.ndarray):
        key = (race, gender)
        self.by_subgroup.setdefault(key, {"paths": [], "vecs": []})
        self.by_subgroup[key]["paths"].append(path)
        self.by_subgroup[key]["vecs"].append(norm_vec)

    def build_trees(self):
        for key, d in self.by_subgroup.items():
            d["vecs"] = np.stack(d["vecs"], axis=0)
            d["tree"] = cKDTree(d["vecs"])

    def nearest_source(self, race: str, gender: str, target_vec: np.ndarray,
                        exclude_path: str, k: int = 5):
        """Retorna o path do rosto mais próximo dentro do MESMO subgrupo,
        excluindo o próprio rosto alvo. k>1 para poder pular o próprio item."""
        key = (race, gender)
        if key not in self.by_subgroup:
            return None
        d = self.by_subgroup[key]
        if len(d["paths"]) < 2:
            return None
        k = min(k, len(d["paths"]))
        dists, idxs = d["tree"].query(target_vec, k=k)
        idxs = np.atleast_1d(idxs)
        for idx in idxs:
            candidate = d["paths"][idx]
            if candidate != exclude_path:
                return candidate
        return None


# --------------------------------------------------------------------------
# 3. Blending (composição) dos dois rostos
# --------------------------------------------------------------------------

def convex_hull_mask(image_shape, landmarks: np.ndarray) -> np.ndarray:
    """Máscara binária (0/255) do casco convexo dos landmarks faciais,
    levemente erodida/suavizada para um blending mais natural."""
    mask = np.zeros(image_shape[:2], dtype=np.uint8)
    hull = cv2.convexHull(landmarks.astype(np.int32))
    cv2.fillConvexPoly(mask, hull, 255)
    # suaviza as bordas da máscara (evita corte muito abrupto)
    mask = cv2.GaussianBlur(mask, (15, 15), 10)
    return mask


def blend_faces(target_img: np.ndarray, source_img: np.ndarray,
                 target_landmarks: np.ndarray, source_landmarks: np.ndarray) -> np.ndarray:
    """
    Alinha o rosto fonte sobre o rosto alvo (via transformação afim estimada
    pelos landmarks) e realiza o blending com cv2.seamlessClone
    (Poisson blending), aproximando a técnica de Blended Images.
    """
    h, w = target_img.shape[:2]

    # Estima uma transformação afim (source -> target) usando 3 pontos
    # de referência estáveis (cantos externos dos olhos e ponta do nariz)
    src_pts = source_landmarks[[36, 45, 30]].astype(np.float32)
    dst_pts = target_landmarks[[36, 45, 30]].astype(np.float32)
    M = cv2.getAffineTransform(src_pts, dst_pts)

    warped_source = cv2.warpAffine(
        source_img, M, (w, h),
        borderMode=cv2.BORDER_REFLECT
    )

    mask = convex_hull_mask(target_img.shape, target_landmarks)
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        raise ValueError('Máscara vazia')

    center = (int(xs.mean()), int(ys.mean()))

    try:
        blended = cv2.seamlessClone(
            warped_source, target_img, mask, center, cv2.NORMAL_CLONE
        )
    except cv2.error:
        # fallback: blending alfa simples se seamlessClone falhar
        alpha = (mask.astype(np.float32) / 255.0)[..., None]
        blended = (warped_source.astype(np.float32) * alpha +
                   target_img.astype(np.float32) * (1 - alpha)).astype(np.uint8)

    return blended


## 2. Gerar pares e registrar proveniência
Comece com 10 pares por subgrupo e split para inspecionar qualidade; depois escolha uma nova pasta de execução e aumente o limite. A rotina não garante contagens iguais se houver falhas: confira a tabela final. Fonte e alvo nunca cruzam treino/validação/teste.


In [ ]:
RUN='piloto_10'
MAX_PAIRS=10  # após inspeção: por exemplo 200, usando outra RUN
OUT=ROOT/'geracoes'/RUN
assert not (OUT/'dataset.csv').exists(), 'Esta execução já existe. Use outra RUN.'
OUT.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(ROOT/'metadata_reais.csv')
assert df.image_id.is_unique and set(df.split)=={'train','val','test'}
assert df.groupby('unit_id').split.nunique().max()==1
ext=LandmarkExtractor(str(predictor)); rows=[]; failures=[]
for (split,race,gender),group in df.groupby(['split','race','gender']):
    valid=[]
    for _,r in tqdm(group.iterrows(),total=len(group),desc=f'{split}/{race}/{gender}'):
        im=cv2.imread(r.image_path)
        if im is None:
            failures.append({'image_id':r.image_id,'reason':'leitura'});continue
        lm,_=ext.extract(im)
        if lm is None:
            failures.append({'image_id':r.image_id,'reason':'sem rosto único'});continue
        valid.append((r,lm))
    np.random.default_rng(42).shuffle(valid)  # evitar escolher alvos pela ordem de idade/nome
    count=0
    for i,(target,tlm) in enumerate(valid):
        if count>=MAX_PAIRS:break
        candidates=[(j,np.linalg.norm(normalize_landmarks(tlm)-normalize_landmarks(lm)))
                    for j,(r,lm) in enumerate(valid) if j!=i and r.unit_id!=target.unit_id]
        if not candidates:continue
        j=min(candidates,key=lambda x:x[1])[0]; source,slm=valid[j]
        try:
            ti=cv2.imread(target.image_path);si=cv2.imread(source.image_path)
            fake=blend_faces(ti,si,tlm,slm)
            if np.mean(np.abs(fake.astype(float)-ti.astype(float)))<.1:
                raise ValueError('Manipulação praticamente idêntica')
            pair=target.image_id
            # Ambos salvos com o mesmo tamanho/codec: evitar atalho por compressão.
            realpath=OUT/f'{pair}_real.jpg';fakepath=OUT/f'{pair}_fake.jpg'
            assert cv2.imwrite(str(realpath),ti,[cv2.IMWRITE_JPEG_QUALITY,95])
            assert cv2.imwrite(str(fakepath),fake,[cv2.IMWRITE_JPEG_QUALITY,95])
            base=target.to_dict();base.update(pair_id=pair,target_id=target.image_id,
                source_id=source.image_id,source_unit_id=source.unit_id,
                generation_method='affine_poisson_blending_v2')
            for label,path in [(0,realpath),(1,fakepath)]:
                row=base.copy();row.update(label=label,image_id=f'{pair}_{label}',image_path=str(path),filename=path.name)
                rows.append(row)
            count+=1
        except Exception as e:failures.append({'image_id':target.image_id,'reason':str(e)})
out=pd.DataFrame(rows)
assert len(out), 'Nenhum par gerado; examine as falhas e os landmarks.'
out.to_csv(OUT/'dataset.csv',index=False)
pd.DataFrame(failures).to_csv(OUT/'falhas.csv',index=False)
print('Use este caminho no notebook 02:',OUT/'dataset.csv')
display(out.groupby(['split','race','gender','label']).size().rename('n').to_frame())


## 3. Inspeção de qualidade
Amostras aleatórias por grupo devem ser revisadas antes de treinar. Não selecione pela dificuldade para o detector. Para a análise confirmatória, use base independente com identidades e métodos de geração documentados.


In [ ]:
import matplotlib.pyplot as plt
sample=out[out.label==1].groupby(['race','gender'],group_keys=False).sample(n=1,random_state=42)
fig,axes=plt.subplots(len(sample),2,figsize=(7,3*len(sample)),squeeze=False)
for axs,(_,r) in zip(axes,sample.iterrows()):
    real=out[(out.pair_id==r.pair_id)&(out.label==0)].iloc[0]
    for ax,p,title in zip(axs,[real.image_path,r.image_path],[f'{r.race} / {r.gender}: real','Manipulado']):
        ax.imshow(cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB));ax.set_title(title);ax.axis('off')
plt.tight_layout();plt.savefig(OUT/'inspecao.png',dpi=120);plt.show()
